In [2]:
from airflow.sdk import dag, task
from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.standard.operators.bash import BashOperator
from airflow.sdk import Param
from airflow.sdk.definitions.param import ParamsDict
from sqlalchemy import create_engine, VARCHAR, Integer,Date, inspect
import os
import pandas as pd
from datetime import datetime
import zipfile
import numpy as np
import requests
from datetime import datetime
from pathlib import Path
import shutil
import boto3
from botocore.config import Config


In [3]:
def unzip_file(zip_path, file_save):
    os.makedirs(file_save, exist_ok=True)

    for file_name in os.listdir(zip_path):
        if file_name.lower().endswith('.zip'):
            zip_file = os.path.join(zip_path, file_name)
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                for member in zip_ref.namelist():
                    nome_arquivo = os.path.basename(member)
                    destino_final = os.path.join(file_save, nome_arquivo)
                    with zip_ref.open(member) as origem, open(destino_final, 'wb') as destino:
                        shutil.copyfileobj(origem, destino)
                    print(f"Extraído: {nome_arquivo}")

In [ ]:
def local_votacao (path):
    colunas_leitura = [
      'dt_geracao', 'hh_geracao', 'aa_eleicao', 'dt_eleicao', 'ds_eleicao',
       'nr_turno', 'sg_uf', 'cd_municipio', 'nm_municipio', 'nr_zona',
       'nr_secao', 'cd_tipo_secao_agregada', 'ds_tipo_secao_agregada',
       'nr_secao_principal', 'nr_local_votacao', 'nm_local_votacao',
       'cd_tipo_local', 'ds_tipo_local', 'ds_endereco', 'nm_bairro', 'nr_cep',
       'nr_telefone_local', 'nr_latitude', 'nr_longitude',
       'cd_situ_local_votacao', 'ds_situ_local_votacao', 'cd_situ_zona',
       'ds_situ_zona', 'cd_situ_secao', 'ds_situ_secao', 'cd_situ_localidade',
       'ds_situ_localidade', 'cd_situ_secao_acessibilidade',
       'ds_situ_secao_acessibilidade', 'qt_eleitor_secao',
       'qt_eleitor_eleicao_federal', 'qt_eleitor_eleicao_estadual',
       'qt_eleitor_eleicao_municipal', 'nr_local_votacao_original',
       'nm_local_votacao_original', 'ds_endereco_locvt_original',
       'id_zona_secao_municipio'
    ]
    dfs = []
    colunas_num = ['qt_eleitor_secao', 'qt_eleitor_eleicao_federal', 'qt_eleitor_eleicao_estadual','qt_eleitor_eleicao_municipal']
    colunas_data = ['dt_geracao','dt_eleicao']
    colunas_hora = ['hh_geracao']
    for bweb in os.listdir(path):
        full_path = os.path.join(path, bweb)
        if full_path.endswith('.csv'):
            print(f"Lendo o arquivo: {full_path}")
            df = pd.read_csv(full_path, encoding='latin-1', sep=';',dtype=str,usecols=lambda col: col.strip().lower() in colunas_leitura, nrows=10)
            df.columns = [col.strip().lower() for col in df.columns]
            for col in colunas_num:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col]).astype('Int64')
            for col in colunas_data:
                if col in df.columns:
                    df[col] = pd.to_datetime(df[col], dayfirst=True, format='%d/%m/%Y', errors='coerce').dt.date
            for col in colunas_hora:
                if col in df.columns:
                    df[col] = pd.to_datetime(df[col], format='%H:%M:%S', errors='coerce').dt.time
            df['aa_eleicao'] = df['aa_eleicao'].astype(str).str.lstrip("0")
            df['id_zona_secao_municipio'] = df['aa_eleicao'] +  df['nr_turno'] + df['nr_zona'] +  df['nr_secao'] +  df['cd_municipio']
            dfs.append(df)


    return df

In [34]:
etl_local_votacao_RR = local_votacao(path = r'C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\local_votacao\csv')

Lendo o arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\local_votacao\csv\eleitorado_local_votacao_2020.csv
Lendo o arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\local_votacao\csv\eleitorado_local_votacao_2022.csv
Lendo o arquivo: C:\Users\Nelio\OneDrive\TRE2026\Archives\RR\local_votacao\csv\eleitorado_local_votacao_2024.csv


In [32]:
etl_local_votacao_RR

,dt_geracao,hh_geracao,aa_eleicao,dt_eleicao,ds_eleicao,nr_turno,sg_uf,cd_municipio,nm_municipio,nr_zona,...,cd_situ_secao_acessibilidade,ds_situ_secao_acessibilidade,qt_eleitor_secao,qt_eleitor_eleicao_federal,qt_eleitor_eleicao_estadual,qt_eleitor_eleicao_municipal,nr_local_votacao_original,nm_local_votacao_original,ds_endereco_locvt_original,id_zona_secao_municipio
0,2024-10-29,02:00:30,2024,2024-10-06,1º Turno,1,BA,39730,LAPÃO,104,...,1,Com acessibilidade,262,0,0,262,1147,ESCOLA DOM PEDRO I,POVOADO DE CASAL I - ZONA RURAL,202411046239730
1,2024-10-29,02:00:30,2024,2024-10-06,1º Turno,1,BA,39730,LAPÃO,104,...,1,Com acessibilidade,307,0,0,306,1457,ESCOLA MUNICIPAL ZENÁLIA DOURADO LOPES,"RUA JOSÉ VILELA, S/N - ZONA URBANA",2024110418139730
2,2024-10-29,02:00:30,2024,2024-10-06,1º Turno,1,SC,83798,VIDEIRA,36,...,0,Sem acessibilidade,388,0,0,388,1147,FUNOESC - FUNDAÇÃO UNIVERSIDADE DO OESTE DE SA...,"RUA PAESE, 198",202413610383798
3,2024-10-29,02:00:30,2024,2024-10-06,1º Turno,1,SC,83798,VIDEIRA,36,...,0,Sem acessibilidade,271,0,0,271,1015,ESCOLA DE EDUCAÇÃO BÁSICA PROFESSORA ADELINA R...,"RUA XV DE NOVEMBRO, N. 683",20241369583798
4,2024-10-29,02:00:30,2024,2024-10-06,1º Turno,1,SP,72451,VOTUPORANGA,147,...,0,Sem acessibilidade,250,0,0,250,1074,EE PROFA SARA ARNOLD BARBOSA,"R RIO GRANDE, 3802",2024114717872451
5,2024-10-29,02:00:30,2024,2024-10-06,1º Turno,1,SC,83798,VIDEIRA,36,...,1,Com acessibilidade,278,0,0,278,1015,ESCOLA DE EDUCAÇÃO BÁSICA PROFESSORA ADELINA R...,"RUA XV DE NOVEMBRO, N. 683",20241362583798
6,2024-10-29,02:00:30,2024,2024-10-06,1º Turno,1,SC,83798,VIDEIRA,36,...,0,Sem acessibilidade,369,0,0,369,1350,ESCOLA DE EDUCAÇÃO BÁSICA MUNICIPAL FIDELIS AN...,"RUA PRESIDENTE CASTELO BRANCO, N. 55",20241367783798
7,2024-10-29,02:00:30,2024,2024-10-06,1º Turno,1,BA,39748,MADRE DE DEUS,162,...,1,Com acessibilidade,366,0,0,366,1058,ESCOLA NOSSA SENHORA DE MADRE DE DEUS,"RUA A, S/N - ZONA URBANA",2024116212039748
8,2024-10-29,02:00:30,2024,2024-10-06,1º Turno,1,SP,72451,VOTUPORANGA,147,...,0,Sem acessibilidade,281,0,0,281,1120,CEM PROF. FAUSTINO PEDROSO,"RUA VILA RICA, 2943",2024114716872451
9,2024-10-29,02:00:30,2024,2024-10-06,1º Turno,1,SC,83836,XANXERÊ,43,...,0,Sem acessibilidade,294,0,0,294,1503,ESCOLA MUNICIPAL DE EDUCAÇÃO BÁSICA CIRILO DAL...,"RUA DOSOLINO CAVAGNOLLI, 500",202414312783836


In [11]:
etl_local_votacao_RR.columns

Index(['dt_geracao', 'hh_geracao', 'aa_eleicao', 'dt_eleicao', 'ds_eleicao',
       'nr_turno', 'sg_uf', 'cd_municipio', 'nm_municipio', 'nr_zona',
       'nr_secao', 'cd_tipo_secao_agregada', 'ds_tipo_secao_agregada',
       'nr_secao_principal', 'nr_local_votacao', 'nm_local_votacao',
       'cd_tipo_local', 'ds_tipo_local', 'ds_endereco', 'nm_bairro', 'nr_cep',
       'nr_telefone_local', 'nr_latitude', 'nr_longitude',
       'cd_situ_local_votacao', 'ds_situ_local_votacao', 'cd_situ_zona',
       'ds_situ_zona', 'cd_situ_secao', 'ds_situ_secao', 'cd_situ_localidade',
       'ds_situ_localidade', 'cd_situ_secao_acessibilidade',
       'ds_situ_secao_acessibilidade', 'qt_eleitor_secao',
       'qt_eleitor_eleicao_federal', 'qt_eleitor_eleicao_estadual',
       'qt_eleitor_eleicao_municipal', 'nr_local_votacao_original',
       'nm_local_votacao_original', 'ds_endereco_locvt_original',
       'id_zona_secao_municipio'],
      dtype='object')